# Ретроспективная проверка Hutton и Полякова на VAAD

Единица анализа — `field_uid × season`. Первое зарегистрированное появление болезни задаётся интервалом `(последний отрицательный визит, первый положительный визит]`. Основной срез использует только прямые геопривязки; восстановленные однозначные привязки добавляются как анализ чувствительности.

Период 2015–2022 используется как development, 2023–2025 — как temporal holdout. Сезон 2026 исключён как незавершённый. Отрицательные сезоны не считаются доказанными true negative без протокола осмотра и данных об обработках.

In [ ]:
from pathlib import Path
import os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    REPO_DIR = Path('/content/AgroPhenology')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/vkonov2/AgroPhenology.git', str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
else:
    REPO_DIR = Path.cwd()
    if not (REPO_DIR / 'pyproject.toml').exists():
        REPO_DIR = Path.cwd().parent

print('Repository:', REPO_DIR.resolve())

In [ ]:
# В Colab файл VAAD не лежит в Git, поэтому по умолчанию откроется загрузка файла.
INPUT_MODE = 'upload' if IN_COLAB else 'local'  # upload | drive | local
LOCAL_CSV_PATH = REPO_DIR / 'data/vaad_observations_geocoded_recovered.csv'
DRIVE_CSV_PATH = '/content/drive/MyDrive/vaad_observations_geocoded_recovered.csv'
MAXIMUM_COMPLETE_SEASON = 2025
BATCH_SIZE = 20
PERMUTATIONS = 5000
OUTPUT_DIR = REPO_DIR / 'results/vaad_late_blight'
CACHE_DIR = REPO_DIR / 'data/cache/vaad_open_meteo'

In [ ]:
if INPUT_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Загрузите ровно один CSV')
    CSV_PATH = Path('/content') / next(iter(uploaded))
elif INPUT_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    CSV_PATH = Path(DRIVE_CSV_PATH)
elif INPUT_MODE == 'local':
    CSV_PATH = Path(LOCAL_CSV_PATH)
else:
    raise ValueError(f'Неизвестный INPUT_MODE: {INPUT_MODE}')

if not CSV_PATH.exists():
    raise FileNotFoundError(CSV_PATH)
print('Input:', CSV_PATH.resolve(), f'({CSV_PATH.stat().st_size / 1024**2:.1f} MB)')

In [ ]:
from agro_phenology.vaad_validation import run_vaad_validation

summary = run_vaad_validation(
    CSV_PATH,
    output_dir=OUTPUT_DIR,
    cache_dir=CACHE_DIR,
    maximum_season=MAXIMUM_COMPLETE_SEASON,
    batch_size=BATCH_SIZE,
    repetitions=PERMUTATIONS,
)
print('SHA-256:', summary['input']['sha256'])

In [ ]:
import pandas as pd

rows = []
for mode in ('direct', 'expanded'):
    cohort = summary['hutton'][mode]['holdout_2023_2025_unseen_fields']
    for lookback in (7, 14, 21):
        metric = cohort[f'lookback_{lookback}d']
        rows.append({
            'model': 'Hutton',
            'mode': mode,
            'design': f'{lookback}-day lookback',
            'n': metric['first_detection']['n'],
            'hits': metric['first_detection']['successes'],
            'hit_rate': metric['first_detection']['rate'],
            'calendar_null': metric['date_permutation']['null_mean_rate'],
            'lift': metric['date_permutation']['lift'],
            'permutation_p': metric['date_permutation']['permutation_p_one_sided'],
            'alarm_day_fraction': metric['alarm_burden_june_august']['alarm_day_fraction'],
        })
    poly = summary['polyakov'][mode]['holdout_2023_2025_unseen_fields']
    for design in ('observed_bbch51_point_assumption', 'exact_bbch51_interval_le_21d', 'phenology_and_onset_le_21d'):
        metric = poly[design]
        rows.append({
            'model': 'Polyakov',
            'mode': mode,
            'design': design,
            'n': metric['possible']['n'],
            'hits': metric['possible']['successes'],
            'hit_rate': metric['possible']['rate'],
            'calendar_null': metric['date_permutation']['null_mean_rate'],
            'lift': metric['date_permutation']['lift'],
            'permutation_p': metric['date_permutation']['permutation_p_one_sided'],
            'alarm_day_fraction': None,
        })
result_table = pd.DataFrame(rows)
display_table = result_table.copy()
for column in ('hit_rate', 'calendar_null', 'alarm_day_fraction'):
    display_table[column] = display_table[column].map(lambda x: '—' if pd.isna(x) else f'{x:.1%}')
display_table['lift'] = display_table['lift'].map(lambda x: '—' if pd.isna(x) else f'{x:+.1%}')
display_table['permutation_p'] = display_table['permutation_p'].map(lambda x: '—' if pd.isna(x) else f'{x:.3f}')
display(display_table)

In [ ]:
h = summary['hutton']['direct']['holdout_2023_2025_unseen_fields']
p = summary['polyakov']['direct']['holdout_2023_2025_unseen_fields']
print('ИТОГ')
print('Hutton: высокая доля попаданий, но нет значимого выигрыша над календарной перестановкой;')
print('         модель не подтверждена как самостоятельный классификатор поля.')
print('Polyakov: observed-BBCH51 point holdout:',
      f"{p['observed_bbch51_point_assumption']['possible']['successes']}/"
      f"{p['observed_bbch51_point_assumption']['possible']['n']} попаданий;",
      'строгий bracketed exact-BBCH51:',
      f"{p['exact_bbch51_interval_le_21d']['possible']['successes']}/"
      f"{p['exact_bbch51_interval_le_21d']['possible']['n']}.")
print('         Подтверждения прогностической полезности нет.')
print('Negative-only сезоны используются только как sensitivity analysis, не как доказанные TN.')

In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('Результаты:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)